# VAE 架构源码解析

VAE 的核心架构如下图所示:

<div align="center">
    <img src="./imgs/VAE_architecture.png" alt="VAE Architecture" style="width: 550px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/64485020)

[VAE 论文](https://arxiv.org/pdf/1312.6114v10)

[VAE 模型 Pytorch 实现](https://github.com/pytorch/examples/blob/main/vae/main.py)

## 何为 Auto Encoder?
**Auto Encoder(自编码器，下文简称 AE)**，就是对一个原始表征$x$层层降维，最后得到一个低维稠密的潜在空间(latent space)的表示$z$，然后再通过一个通常是对称的解码器(Decoder)一步步解码得到原始的表征$\tilde{x}$，也就是说，我们对原始的特征$x$的特征空间进行压缩，从而”强迫“模型学会原始表征的”最重要的特征表示“，如下图所示:

<div align="center">
    <img src="./imgs/AE_architecture.png" alt="AE Architecture" style="width: 600px; height: auto;">
</div>

那么 AE 的损失函数就显而易见了，就是重建后的图像$\tilde{x}$和原始图像$x$的均方误差:

$$
\mathcal{L} = \frac{1}{n} \sum_{i=1}^n (\tilde{x}_i - x_i)^2
$$

但是 AE 只能对已知给定的图像进行特征提取得到 $z$，再重建回原始图像，而不能在潜在空间中随机采样得到$z$再进行解码得到有意义的图像，即使随机生成一个$z$解码，得到的图像通常也是无意义的，所以不能很好地适配图像生成任务，而下文要介绍的 **Variational Auto Encoder(变分自编码器，下文简称 VAE)**就解决了这个问题

## Variational Auto Encoder
VAE 相对于 AE 的最大改进在于，它不在拘泥于学习固定的潜在空间的一个向量表示$z$，而是学习潜在空间中的一个高斯分布，让训练输入$x$在潜在空间中的向量表示都服从这个高斯分布，这样我们在推理阶段就可以直接从这个分布中采样出$z$，然后再通过解码器解码得到生成图像$\tilde{x}$，下面就来介绍一下原理和实现

### VAE 目标
VAE的损失函数如下，([推导](theory.md)):

$$
\mathcal{L} = \sum_{i=1}^N \left[
    \| \tilde{x}_i - x_i \|^2 + 
    \frac{1}{2} \left( \mu_i^2 + \sigma_i^2 - \log \sigma_i^2 - 1 \right)
\right]
$$

**直观**理解，$\| \tilde{x}_i - x_i \|^2$就是**重建损失**，目的是让生成的图像$\tilde{x}$尽可能和原始图像$x$相似；而$\frac{1}{2} \left( \mu_i^2 + \sigma_i^2 - \log \sigma_i^2 - 1 \right)$是 **KL 散度损失**，因为在图像生成的推理阶段，我们认为潜在空间的$z$符合**标准正态分布$\mathcal{N}(0, I)$**，所以编码器得到的$z$的分布要尽可能和标准正态分布相近

In [8]:
import torch
import torch.nn as nn


def loss_func(recon_x:torch.Tensor, x:torch.Tensor, mu:torch.Tensor, logvar:torch.Tensor):
    # MSE loss
    MSE = nn.functional.mse_loss(recon_x, x, reduction="sum")

    # KL loss
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return MSE + KLD


# -------------------------------------------------------------------------
# 常数项
# -------------------------------------------------------------------------
BATCH_SIZE = 4
IMG_SIZE = 64
LATENT_DIM = 20
INPUT_DIM = 64


x = torch.randn(BATCH_SIZE, IMG_SIZE, IMG_SIZE)
recon_x = torch.randn(BATCH_SIZE, IMG_SIZE, IMG_SIZE)
mu = torch.randn(BATCH_SIZE, LATENT_DIM)
logvar = torch.randn(BATCH_SIZE, LATENT_DIM)

print(f'x, x_recon size: {x.size()}')
print(f'mu, logvar size: {mu.size()}')
loss = loss_func(recon_x, x, mu, logvar)
print(f'loss: {loss}')


x, x_recon size: torch.Size([4, 64, 64])
mu, logvar size: torch.Size([4, 20])
loss: 33070.375


### 重参数化技巧

因为编码器$q_{\phi}(x)$得到的是均值和方差，我们需要在它们构造的正态分布中采样，但是直接进行采样操作**是不可导的**，为了保证梯度的传递，我们需要利用**重参数化**技巧进行采样:

$$
z = \mu + \sigma \odot \epsilon, \epsilon \sim \mathcal{N}(0, I)
$$

这样反向传播就能进行梯度回传了

In [6]:
def reparameterize(mu:torch.Tensor, logvar:torch.Tensor):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)

    return mu + eps * std  # -> [bs, latent_dim]


sample_z = reparameterize(mu, logvar)
print(f'sample_z size: {sample_z.size()}')


sample_z size: torch.Size([4, 20])


### 完整的 VAE 模型参考示例
下面是我的一个**简单** VAE模型 代码实现:

In [ ]:
from torchvision.utils import save_image
import os


class VAE(nn.Module):
    def __init__(self, latent_dim:int = 128, img_size:int = 64, input_dim:int = 64):
        super().__init__()

        self.latent_dim = latent_dim
        self.img_size = img_size
        self.input_dim = input_dim
        self.down_size = img_size//16
        self.flatten_size = input_dim*8*self.down_size*self.down_size

        # [bs, 3, img_size, img_size], range(-1, 1) -> 
        self.encoder = nn.Sequential(   
            nn.Conv2d(3, input_dim, 4, 2, 1),  # -> [bs, input_dim, img_size/2, img_size/2]
            nn.ReLU(),
            nn.Conv2d(input_dim, input_dim*2, 4, 2, 1),  # -> [bs, input_dim*2, img_size/4, img_size/4]
            nn.ReLU(),
            nn.Conv2d(input_dim*2, input_dim*4, 4, 2, 1),  # -> [bs, input_dim*4, img_size/8, img_size/8]
            nn.ReLU(),
            nn.Conv2d(input_dim*4, input_dim*8, 4, 2, 1),  # -> [bs, input_dim*8, img_size/16, img_size/16]
            nn.ReLU(),
            nn.Flatten()  # -> [bs, input_dim*8*(img_size/16)*(img_size/16)]
        )

        # [bs, input_dim*8*(img_size/16)*(img_size/16)] -> [bs, latent_dim]
        self.fc_mu = nn.Linear(self.flatten_size, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_size, latent_dim)

        # -> [bs, input_dim*8, img_size/16, img_size/16]
        self.decoder_input = nn.Linear(latent_dim, self.flatten_size)

        self.decoder = nn.Sequential(
            nn.Unflatten(1, (input_dim*8, self.down_size, self.down_size)),  # -> [bs, input_dim*8, img_size/16, img_size/16]
            nn.ConvTranspose2d(input_dim*8, input_dim*4, 4, 2, 1),  # -> [bs, input_dim*4, img_size/8, img_size/8]
            nn.ReLU(),
            nn.ConvTranspose2d(input_dim*4, input_dim*2, 4, 2, 1),  # -> [bs, input_dim*2, img_size/4, img_size/4]
            nn.ReLU(),
            nn.ConvTranspose2d(input_dim*2, input_dim, 4, 2, 1),  # -> [bs, input_dim, img_size/2, img_size/2]
            nn.ReLU(),
            nn.ConvTranspose2d(input_dim, 3, 4, 2, 1),  # -> [bs, 3, img_size, img_size]
            nn.Tanh()  # -> range(-1, 1)  
        )

    def encode(self, x:torch.Tensor):
        feat = self.encoder(x)  # -> [bs, input_dim*8*(img_size/16)*(img_size/16)]

        # -> [bs, latent_dim] * 2
        return self.fc_mu(feat), self.fc_logvar(feat)
    
    def reparameterize(self, mu:torch.Tensor, logvar:torch.Tensor):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        return mu + eps * std  # -> [bs, latent_dim]
    
    def decode(self, z:torch.Tensor) -> torch.Tensor:
        x = self.decoder_input(z)  # -> [bs, input_dim*8, img_size/16, img_size/16]

        return self.decoder(x)  # -> [bs, 3, img_size, img_size]
    
    def forward(self, x:torch.Tensor):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)

        return self.decode(z), mu, logvar
    
    def print_forward(self, x:torch.Tensor):
        print(f'[input x size]: {x.size()}')

        mu, logvar = self.encode(x)
        print(f'[mu & logvar size]: {mu.size()}')

        z = self.reparameterize(mu, logvar)
        print(f'[reparameterized z size]: {z.size()}')

        output = self.decode(z)
        print(f'[after decoding, output size]: {output.size()}')

    def inference(self, sample_num:int = 4, save_dir:str = './samples'):
        self.eval()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        os.makedirs(save_dir, exist_ok=True)

        with torch.no_grad():
            sample = torch.randn(sample_num, self.latent_dim).to(device)
            sample = self.decode(sample).cpu()

            for i in range(sample_num):
                save_image(
                    sample[i],
                    f"{save_dir}/sample_{i}.png",
                    normalize=False
                )

                print(f"Saved sample_{i}.png to {save_dir}")
                

In [9]:
def test_vae():
    x = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE)
    vae = VAE(LATENT_DIM, IMG_SIZE, INPUT_DIM)
    vae.print_forward(x)


test_vae()


[input x size]: torch.Size([4, 3, 64, 64])
[mu & logvar size]: torch.Size([4, 20])
[reparameterized z size]: torch.Size([4, 20])
[after decoding, output size]: torch.Size([4, 3, 64, 64])


## 示例训练和采样代码
下面是我的训练和采样代码的**简单**实现:

In [10]:
from c_vae import (
    get_config, get_trainer
)


# epochs = 500，示例
# 详细参考: casual_paper_learning/c_vae/train.py & ./configs/vae_config.yaml
config = get_config('./configs/vae_config.yaml')
trainer = get_trainer(config)
trainer.train()


Training: 100%|██████████| 500/500 [00:14<00:00, 34.15it/s, avg_loss=93.5]  


[trainer]Saved: vae_iter500.pth


In [16]:
from c_vae import (
    get_config, get_vae
)


# 加载的ckpt是训练了5000个epochs的
# 详细参考: ./configs/vae_config.yaml
config = get_config('./configs/vae_config.yaml')
vae_pretrained = get_vae(config, load_pretrained=True)
vae_pretrained.inference(sample_num=4)


[VAE]Loaded checkpoint from ./ckpts/vae_iter5000.pth
Saved sample_0.png to ./samples
Saved sample_1.png to ./samples
Saved sample_2.png to ./samples
Saved sample_3.png to ./samples


可以看看效果(很一般):

<div style="display: flex; justify-content: center; gap: 12px; align-items: center; margin: 16px 0;">
  <div style="flex: 1; max-width: 120px; text-align: center;">
    <img src="./samples/sample_0.png" style="width: 100%; height: auto;" alt="img1">
    <p style="margin: 4px 0 0; font-size: 14px;">图 1</p>
  </div>
  <div style="flex: 1; max-width: 120px; text-align: center;">
    <img src="./samples/sample_1.png" style="width: 100%; height: auto;" alt="img2">
    <p style="margin: 4px 0 0; font-size: 14px;">图 2</p>
  </div>
  <div style="flex: 1; max-width: 120px; text-align: center;">
    <img src="./samples/sample_2.png" style="width: 100%; height: auto;" alt="img3">
    <p style="margin: 4px 0 0; font-size: 14px;">图 3</p>
  </div>
  <div style="flex: 1; max-width: 120px; text-align: center;">
    <img src="./samples/sample_3.png" style="width: 100%; height: auto;" alt="img4">
    <p style="margin: 4px 0 0; font-size: 14px;">图 4</p>
  </div>
</div>

(第二张叠加态🐧快笑死我了)